# Буткемп, день 14 — стоимость как компоненты движка

Задача дня сформулирована коротко: **назначить цену каждой издержке и сложить их без остатка**. Вчерашний движок знал, когда заявка исполняется; сегодня он должен знать, сколько стоит каждый филл: комиссия, оставленный след, funding, перевод маржи между площадками. Издержки — не константа, которую вычитают в конце. Комиссия зависит от объёма за предыдущие 30 дней, impact — от того, как разбит ордер, funding — от позиции в конкретный момент, а хеджированная позиция на двух площадках — это две позиции, каждой из которых нужна маржа.

Ноутбук устроен как набор компонентов с общей книгой (ledger). У каждого компонента своё состояние и своя строка. В конце строится waterfall, где каждая строка помечена как **measured** (измерено по L2), **cited** (взято из расписания/слайда; в реальной сдаче — со ссылкой на биржу и датой) или **assumed** (модельное допущение, которое проверяется sweep'ом). Тождество сохранения проверяется отдельно: изменение капитала равно ценовому P&L плюс сумме всех строк книги.

Данные — те же, что в днях 11–13: 100-мс top-of-book по BTC на Bybit, Okex и Binance и полные стаканы Bybit/Okex. Для replay берётся окно 2026-07-01 12:00–12:10 UTC: десяти минут достаточно, чтобы получить сотни филлов и измерить кривую стакана, но мало для сканирования гигабайтов. Funding и 24-часовые пути позиции моделируются отдельно по расписанию — они не требуют непрерывного replay. Полные стаканы читаются короткими срезами по 2 минуты; это единственный способ не сканировать 6.7 ГБ целиком.

## Настройка и разведка данных

Сначала фиксируем окно и проверяем, что именно даёт 100-мс файл: спред, глубину на touch и расхождение цен между площадками. Сразу видно масштаб: медианный спред BTC на всех трёх биржах — около $0.018$ bp, то есть меньше тысячной доли процента. Это важная отправная точка: котировка «на touch» зарабатывает половину этого спреда, а комиссии ниже измеряются в десятых долях bp. Поэтому широкая котировка из модели дня 12 ($1.167$ bp) — это **допущение о политике**, а не измеренный рынок; в waterfall это будет помечено явно.

In [1]:
import numpy as np, pandas as pd, duckdb, heapq, hashlib, math, warnings
from collections import defaultdict
warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

D   = "/Users/b.kotov/Downloads/"
Q   = D + "l2_executable_quotes_100ms_10000.parquet"
FULL = {"Bybit": (D + "bybit_l2_20260701_20260703.parquet", "BTCUSDT"),
        "Okex":  (D + "okex_l2_20260701_20260703.parquet",  "BTC-USDT-SWAP"),
        "Binance": (D + "binance_l2_20260701_20260703.parquet", "BTCUSDT")}
WIN_LO, WIN_HI = "2026-07-01 12:00:00+00", "2026-07-01 12:10:00+00"
con = duckdb.connect()

print("файлы:")
for name, (path, _) in FULL.items():
    print(f"  {name:8s} {path.split('/')[-1]:42s} {__import__('os').path.getsize(path)/1e9:5.2f} GB")
print(f"  top-of-book {Q.split('/')[-1]:42s} {__import__('os').path.getsize(Q)/1e9:5.2f} GB")

файлы:
  Bybit    bybit_l2_20260701_20260703.parquet          1.37 GB
  Okex     okex_l2_20260701_20260703.parquet           3.91 GB
  Binance  binance_l2_20260701_20260703.parquet        7.24 GB
  top-of-book l2_executable_quotes_100ms_10000.parquet    0.24 GB


In [2]:
# 100-мс панель: BTC, три площадки, 10 минут
dfq = con.execute(f"""
    SELECT exchange, event_time, mid, best_bid, best_ask,
           best_bid_amount, best_ask_amount
    FROM read_parquet('{Q}')
    WHERE asset='BTC'
      AND event_time >= TIMESTAMPTZ '{WIN_LO}'
      AND event_time <  TIMESTAMPTZ '{WIN_HI}'
    ORDER BY event_time
""").df()
dfq["t"] = pd.to_datetime(dfq.event_time, utc=True).dt.floor("100ms")

g = dfq.groupby(["exchange", "t"]).agg(
        mid=("mid", "last"), bb=("best_bid", "last"), ba=("best_ask", "last"),
        bba=("best_bid_amount", "last"), baa=("best_ask_amount", "last")).reset_index()

def pivot_col(col):
    return g.pivot(index="t", columns="exchange", values=col).sort_index().ffill().dropna()

MID, BB, BA = pivot_col("mid"), pivot_col("bb"), pivot_col("ba")
BBA, BAA = pivot_col("bba"), pivot_col("baa")
common = MID.index
for x in (BB, BA, BBA, BAA):
    common = common.intersection(x.index)
MID, BB, BA = MID.loc[common], BB.loc[common], BA.loc[common]
BBA, BAA = BBA.loc[common], BAA.loc[common]
idx_full = common
idx = idx_full[:-1]                       # последний шаг нужен только для markout
midB = MID["Binance"].to_numpy()[:-1]
midY = MID["Bybit"].to_numpy()[:-1]
midO = MID["Okex"].to_numpy()[:-1]
microY = ((BA["Bybit"].to_numpy()*BBA["Bybit"].to_numpy() +
           BB["Bybit"].to_numpy()*BAA["Bybit"].to_numpy()) /
          (BBA["Bybit"].to_numpy()+BAA["Bybit"].to_numpy()))[:-1]
NSTEP = len(midB)

spread = pd.DataFrame({ex: (BA[ex]-BB[ex])/MID[ex]*1e4 for ex in MID.columns})
basis = pd.DataFrame({"Okex": (MID["Okex"]-MID["Bybit"])/MID["Bybit"]*1e4,
                      "Binance": (MID["Binance"]-MID["Bybit"])/MID["Bybit"]*1e4})
print(f"панель: {NSTEP:,} шагов по 100 мс  |  {idx[0]}  ->  {idx[-1]}")
print("\nмедианный спред, bp:")
print(spread.median().round(4).to_string())
print("\nbasis к Bybit, bp (mean / sd):")
print(pd.DataFrame({"mean": basis.mean(), "sd": basis.std()}).round(4).to_string())
print(f"\nсредняя цена BTC за окно: ${MID['Bybit'].mean():,.0f}")

панель: 5,999 шагов по 100 мс  |  2026-07-01 12:00:00+00:00  ->  2026-07-01 12:09:59.800000+00:00

медианный спред, bp:
Binance    0.0171
Bybit      0.0171
Okex       0.0171

basis к Bybit, bp (mean / sd):
           mean      sd
Okex    -0.5623  0.4027
Binance  0.1669  0.3849

средняя цена BTC за окно: $58,524


Спред на всех трёх площадках — около $0.018$ bp, то есть примерно один тик $0.1$ при цене $\approx 60{,}000$. Половина тика — $0.009$ bp. Это число дальше будет играть роль «честной» maker-выручки, если стоять на touch. Модель дня 12 ставит котировку на $1.167$ bp от середины: это не измерение, а выбранная политика с параметрами $k=1.5$ и стоимостью хеджа $0.5$ bp. Водопад нужно читать с этой оговоркой: положительный gross edge здесь — свойство модели, а не рынка.

Гипотеза дня: **gross edge $1.167$ bp переживёт комиссии только в верхнем тарифном тире, где maker получает rebate; при тарифе без rebate и реалистичном adverse selection запас схлопывается почти в ноль, а при котировке на реальном touch — становится отрицательным.** Проверять это будем не одним числом, а разложением по строкам и sensitivity-таблицей.

## Компоненты: книга и комиссии

Начнём с того, что не требует стакана. Комиссия — ступенчатая функция объёма за **прошлые** 30 дней. В книге это отдельная строка, которая пишется на каждом филле; состояние — trailing volume, известный **до** филла. Рядом лежат две ловушки, которые надо посчитать, а не запомнить: обрыв у порога (cliff) и look-ahead, когда backtest ставит сегодняшнему филлу тариф, заработанный только будущим объёмом.

Расписание ниже — иллюстративное, взято со слайда 6 и помечено `cited/illustrative`. В сдаче на реальные деньги его заменяют тарифной сеткой биржи с датой. Для maker: ниже \$1B — $+0.5$ bp, \$1–5B — $0$, выше \$5B — $-0.5$ bp (rebate). Для taker добавлена такая же иллюстративная лестница, потому что хедж на другой площадке может быть агрессивным.

In [3]:
# --- тарифные сетки (иллюстративные, cited) ---
MAKER_TIERS = [(0.0, 1e9, 0.5), (1e9, 5e9, 0.0), (5e9, np.inf, -0.5)]
TAKER_TIERS = [(0.0, 1e9, 2.5), (1e9, 5e9, 2.0), (5e9, np.inf, 1.5)]

def fee_rate_bp(volume, tiers=MAKER_TIERS):
    for lo, hi, r in tiers:
        if volume < hi:
            return r
    return tiers[-1][2]

class TrailingVolume:
    """Объём за предыдущие window_days; событие с меткой t в объём ДО t не входит."""
    def __init__(self, window_days=30):
        self.w, self.events = float(window_days), []
    def add(self, t, notional):
        self.events.append((float(t), float(notional)))
        self.events.sort()
    def volume_before(self, t):
        t = float(t); lo = t - self.w
        return sum(v for tt, v in self.events if lo <= tt < t)

print("maker (bp):", [f"<${lo/1e9:.0f}B {r:+.1f}" if np.isfinite(hi) else f">${lo/1e9:.0f}B {r:+.1f}" for lo,hi,r in MAKER_TIERS])
print("taker (bp):", [f"<${lo/1e9:.0f}B {r:+.1f}" if np.isfinite(hi) else f">${lo/1e9:.0f}B {r:+.1f}" for lo,hi,r in TAKER_TIERS])

# --- cliff: 0.5 bp скидка при переходе через $1B ---
V0 = np.array([0.50, 0.80, 0.90, 0.95, 0.99])   # $B
DV = np.array([0.50, 0.20, 0.10, 0.05, 0.01])   # $B
CLIFF = pd.DataFrame({"V0, $B": V0, "dV, $B": DV})
CLIFF["fee drop, bp"] = 0.5
CLIFF["saving, $"] = 0.5e-4 * (V0 + DV) * 1e9
CLIFF["break-even loss, bp"] = 0.5 * (V0 + DV) / DV
CLIFF["break-even loss, $"] = CLIFF["break-even loss, bp"] * DV * 1e9 * 1e-4
print("\ncliff: скидка 0.5 bp на весь объём (V0+dV) и цена лишнего объёма")
print(CLIFF.round(2).to_string(index=False))
assert np.allclose(CLIFF["break-even loss, bp"], [1.0, 2.5, 5.0, 10.0, 50.0])
assert np.allclose(CLIFF["saving, $"], 50_000.0)

# --- look-ahead: ramp объёма за 90 дней, lagged vs whole-sample tier ---
def fee_ramp(m_start, m_end, days=90):
    months = np.linspace(m_start, m_end, days)       # $B в месяц
    daily = months / 30 * 1e9
    hist = list(np.full(30, m_start / 30 * 1e9))     # стартовая история
    lag = whole = 0.0
    avg_month = months.mean() * 1e9
    for d in range(days):
        lag += fee_rate_bp(sum(hist[-30:])) * 1e-4 * daily[d]
        whole += fee_rate_bp(avg_month) * 1e-4 * daily[d]
        hist.append(daily[d])
    return dict(lagged=lag, whole=whole, volume=daily.sum(), avg_month=avg_month)

RA, RB = fee_ramp(0.3, 1.5), fee_ramp(0.6, 1.8)
LA = pd.DataFrame([
    ["0.3 -> 1.5 (avg 0.9, ниже порога)", RA["lagged"], RA["whole"], RA["whole"]/RA["lagged"]-1],
    ["0.6 -> 1.8 (avg 1.2, выше порога)", RB["lagged"], RB["whole"], (RB["whole"]/RB["lagged"]-1 if RB["lagged"] else np.nan)],
], columns=["ramp, $B/мес", "lagged, $", "whole-sample, $", "ошибка whole/lagged - 1"])
print("\nlook-ahead на ramp объёма:")
print(LA.assign(**{"lagged, $": LA["lagged, $"].round(0), "whole-sample, $": LA["whole-sample, $"].round(0),
                   "ошибка whole/lagged - 1": (LA["ошибка whole/lagged - 1"]*100).round(1)}).to_string(index=False))
assert abs(RA["lagged"] - 85191) < 1 and abs(RA["whole"] - 135000) < 1
assert abs(RB["lagged"] - 69258) < 1 and RB["whole"] == 0.0
print("  интерпретация: в первом ramp whole-sample завышает комиссии на 58.5%; во втором — стирает их полностью.")

# --- тест fee-tier lag ---
tv = TrailingVolume(30)
tv.add(0.0, 0.9e9)                                  # $0.9B за прошлые 30 дней
r_before = fee_rate_bp(tv.volume_before(0.5))
tv.add(0.4, 0.2e9)                                  # объём вырос до $1.1B
r_old = fee_rate_bp(tv.volume_before(0.3))          # филл ДО объёма
r_new = fee_rate_bp(tv.volume_before(0.5))          # филл ПОСЛЕ объёма
tv.add(1.0, 5.0e9)
r_same_ts = fee_rate_bp(tv.volume_before(1.0))      # ровно в метку: новый объём ещё не виден
r_after = fee_rate_bp(tv.volume_before(1.1))
print(f"\nfee-tier lag: до изменения {r_before:+.1f} bp, до/после изменения {r_old:+.1f}/{r_new:+.1f} bp, "
      f"в ту же метку {r_same_ts:+.1f} bp, после {r_after:+.1f} bp")
assert r_before == 0.5 and r_old == 0.5 and r_new == 0.0 and r_same_ts == 0.0 and r_after == -0.5
FEE_TEST = True

maker (bp): ['<$0B +0.5', '<$1B +0.0', '>$5B -0.5']
taker (bp): ['<$0B +2.5', '<$1B +2.0', '>$5B +1.5']

cliff: скидка 0.5 bp на весь объём (V0+dV) и цена лишнего объёма
 V0, $B  dV, $B  fee drop, bp  saving, $  break-even loss, bp  break-even loss, $
   0.50    0.50           0.5    50000.0                  1.0             50000.0
   0.80    0.20           0.5    50000.0                  2.5             50000.0
   0.90    0.10           0.5    50000.0                  5.0             50000.0
   0.95    0.05           0.5    50000.0                 10.0             50000.0
   0.99    0.01           0.5    50000.0                 50.0             50000.0

look-ahead на ramp объёма:
                     ramp, $B/мес  lagged, $  whole-sample, $  ошибка whole/lagged - 1
0.3 -> 1.5 (avg 0.9, ниже порога)    85191.0         135000.0                     58.5
0.6 -> 1.8 (avg 1.2, выше порога)    69258.0              0.0                   -100.0
  интерпретация: в первом ramp whole-sample завыш

Числа ramp'а совпадают со слайдом: $85{,}191$ против $135{,}000$ в первом прогоне (whole-sample завышает на $58.5\%$) и $69{,}258$ против $0$ во втором (whole-sample стирает комиссию полностью). Причина одна и та же: тариф — ступенька от **прошлого** объёма, а не от среднего по всему периоду. Стратегия, которая растёт, получает лучший тариф позже; backtest, который выдаёт ей его сразу, ошибается в обе стороны.

Cliff показывает, почему у порога нельзя «доторговать до скидки»: скидка $0.5$ bp на весь месячный объём $1B$ стоит ровно $\$50{,}000$, но если до порога не хватает $\$0.1B$, эти $\$50{,}000$ приходятся на $\$0.1B$ — это $5$ bp потерь на добавленном объёме. При $\Delta V=\$0.01B$ цена добавки — $50$ bp против сэкономленных $0.5$ bp. Искусственный объём ради тира не окупается и, кроме того, нарушает правила биржи.

## Impact: что видно в стакане, а что приходится добавлять

Impact раскладывается на три части. Мгновенная — проход по стакану — **измеряется** прямо из L2. Transient (след, который затухает) и permanent (информация, которая остаётся) в L2 отсутствуют: replay не знает, какие сделки были вашими. Их приходится добавлять моделью-пропагатором
$$G(\tau)=\lambda e^{-\rho\tau},\qquad \text{cost}(q)=\tfrac12\sum_{n,m}q_n q_m G(|t_n-t_m|),$$
где $\lambda$ — мгновенный impact на единицу, $\rho$ — скорость восстановления книги. В дискретном handler'е это ровно
$$D\leftarrow D\,e^{-\rho\Delta t}+\lambda q/2,\qquad \text{impact cost}=q\left(D+\lambda q/2\right),\qquad D\leftarrow D+\lambda q.$$
Диагональ даёт $\tfrac12\lambda q^2$, поэтому для одного блока стоимость равна $\tfrac12 q^\top M q$ с $M_{nm}=G(|t_n-t_m|)$.

Сначала измерим, что даёт стакан: среднюю цену прохода для размеров 1/4/8/12 единиц по Bybit и Okex в четырёх срезах. Потом оценим $\rho$ прокси по восстановлению глубины после резкого падения. Прямого измерения $\lambda$ и $\gamma$ в L2 нет — это будут диапазоны допущений.

In [4]:
# --- срезы стакана по 2 минуты; 4 часа, 2 площадки ---
def read_book_piece(venue, hour):
    path, sym = FULL[venue]
    q = f"""
      WITH win AS (
        SELECT event_time, side, price, amount FROM read_parquet('{path}')
        WHERE symbol='{sym}'
          AND event_time >= TIMESTAMPTZ '2026-07-01 {hour:02d}:00:00+00'
          AND event_time <  TIMESTAMPTZ '2026-07-01 {hour:02d}:02:00+00')
      SELECT side, price, arg_max(amount, event_time) AS amount
      FROM win GROUP BY side, price HAVING amount > 0
    """
    d = con.execute(q).df(); d["venue"], d["hour"] = venue, hour
    return d

def walk(book, side_val, units, mid):
    "средняя цена прохода units единиц; off от mid в bp (>0 — платим спред)"
    b = book[book.side == side_val].sort_values("price", ascending=(side_val < 0))
    rem, cost = float(units), 0.0
    for p, a in zip(b.price.to_numpy(), b.amount.to_numpy()):
        take = min(a, rem); cost += take*p; rem -= take
        if rem <= 1e-9: break
    if rem > 1e-9: return np.nan
    avg = cost/units
    return (avg-mid)/mid*1e4 if side_val < 0 else (mid-avg)/mid*1e4

BOOKS = {}
for venue in ("Bybit", "Okex"):
    for hour in (6, 9, 12, 15):
        try:
            BOOKS[(venue, hour)] = read_book_piece(venue, hour)
        except Exception as e:
            print("book fail", venue, hour, e)

rows = []
for venue in ("Bybit", "Okex"):
    for hour in (6, 9, 12, 15):
        bk = BOOKS[(venue, hour)]
        mid = (bk[bk.side == 1].price.max() + bk[bk.side == -1].price.min()) / 2
        row = {"venue": venue, "hour": f"{hour:02d}:00", "mid": mid}
        for u in (1, 4, 8, 12):
            row[f"buy{u}"] = walk(bk, 1, u, mid)
            row[f"sell{u}"] = walk(bk, -1, u, mid)
        rows.append(row)
BW = pd.DataFrame(rows)
print("book-walk: средний fill vs mid, bp (n/a — в срезе меньше глубины)")
print(BW.round(3).to_string(index=False))

# единичный хедж на Okex: стоимость по часам (для replay)
HC1 = {}
for hour in (6, 9, 12, 15):
    bk = BOOKS[("Okex", hour)]
    mid = (bk[bk.side == 1].price.max() + bk[bk.side == -1].price.min()) / 2
    for s in (1, -1):
        HC1[(hour, s)] = walk(bk, s, 1, mid)
HC1_MED = {s: np.nanmedian([v for (h, ss), v in HC1.items() if ss == s and np.isfinite(v)]) for s in (1, -1)}
for k, v in list(HC1.items()):
    if not np.isfinite(v): HC1[k] = HC1_MED[k[1]]
print("\nokex 1-unit hedge, bp:", {f"{h:02d}:00/{'bid' if s==1 else 'ask'}": round(v,4) for (h,s),v in HC1.items()})

# --- прокси resilience: восстановление глубины Bybit после падения ---
dep = BBA["Bybit"].to_numpy()
rec, i, med_win = [], 1, 100
while i < len(dep)-1:
    prev = np.median(dep[max(0,i-med_win):i])
    if prev > 0 and dep[i] < 0.5*prev:
        j, lim = i+1, min(len(dep), i+600)
        while j < lim and dep[j] < 0.8*prev: j += 1
        if j < lim: rec.append((j-i)*0.1)
        i = j
    i += 1
RHO_PROXY = 1.0/np.median(rec) if rec else 1.0
print(f"\nглубина Bybit: падений >50% — {len(rec)}, медиана восстановления {np.median(rec):.2f} с "
      f"(p25/p75 {np.percentile(rec,25):.1f}/{np.percentile(rec,75):.1f} с) -> rho ~ {RHO_PROXY:.2f} 1/с")
print("  это прокси: падение глубины может быть и сделкой, и отменой; разделить их без tape нельзя.")

book-walk: средний fill vs mid, bp (n/a — в срезе меньше глубины)
venue  hour      mid  buy1  sell1  buy4  sell4  buy8  sell8  buy12  sell12
Bybit 06:00 59058.15 0.669  0.008 0.927  0.008 1.049  0.008  1.193   0.120
Bybit 09:00 58907.75 0.008  0.175 0.008  0.646 0.008  0.906  0.059     NaN
Bybit 12:00 58568.65 0.009  0.009 0.332  0.009   NaN  0.009    NaN   0.260
Bybit 15:00 59587.35 0.008  0.059 0.109  0.671 0.214    NaN  0.252     NaN
 Okex 06:00 59046.45 0.231  0.008 0.696  0.008 0.933  0.008  1.037   0.154
 Okex 09:00 58901.15 0.008  0.474 0.008  1.192 0.008  1.572  0.012   1.864
 Okex 12:00 58565.05 0.009  0.009 0.380  0.009 0.658  0.043  0.863   0.266
 Okex 15:00 59588.15 0.008  0.008 0.008  0.008 0.408  0.114  0.718   0.285

okex 1-unit hedge, bp: {'06:00/bid': np.float64(0.2305), '06:00/ask': np.float64(0.0085), '09:00/bid': np.float64(0.0085), '09:00/ask': np.float64(0.474), '12:00/bid': np.float64(0.0085), '12:00/ask': np.float64(0.0085), '15:00/bid': np.float64(0.0084), '15:

In [5]:
# --- пропагатор: матрица, handler, три расписания ---
def prop_matrix(t, lam, rho):
    t = np.asarray(t, float)
    return lam*np.exp(-rho*np.abs(t[:,None]-t[None,:]))
def cost_quad(q, t, lam, rho):
    return 0.5*q @ prop_matrix(t, lam, rho) @ q
def cost_handler(q, t, lam, rho):
    "тот же handler, что в движке: D затухает, impact = D + lam*q/2, затем D += lam*q"
    D, last, c = 0.0, float(t[0]), 0.0
    for qi, ti in zip(q, t):
        D *= np.exp(-rho*(ti-last)); last = ti
        c += qi*(D + lam*qi/2)
        D += lam*qi
    return c
def sched_block(X, N): q = np.zeros(N); q[0] = X; return q
def sched_equal(X, N): return np.full(N, X/N)
def sched_optimal(X, T, N, rho):
    b = X/(rho*T+2)
    q = np.full(N, (X-2*b)/(N-2)); q[0] = q[-1] = b
    return q

print("расписание: block / equal slices / optimal, стоимость относительно одного блока (lam=1, X=1)")
SW = []
for rhoT in (0.5, 3.0, 20.0):
    N, T = 2001, 1.0
    t = np.linspace(0, T, N)
    block = 0.5
    v_eq = cost_quad(sched_equal(1.0, N), t, 1.0, rhoT)
    v_op = cost_quad(sched_optimal(1.0, T, N, rhoT), t, 1.0, rhoT)
    SW.append((rhoT, v_eq/block, v_op/block, 2/(rhoT+2)))
print(pd.DataFrame(SW, columns=["rho*T", "equal/block", "optimal/block", "2/(rhoT+2)"]).round(4).to_string(index=False))

# known-answer: 60 слотов, rho*T=8, lam=2, X=1
N, T, rho, lam, X = 60, 1.0, 8.0, 2.0, 1.0
t = np.linspace(0, T, N); q = sched_optimal(X, T, N, rho)
c_hand = cost_handler(q, t, lam, rho); c_quad = cost_quad(q, t, lam, rho)
print(f"\nknown-answer (60 слотов, rhoT=8): handler {c_hand:.12f}  quad {c_quad:.12f}  diff {c_hand-c_quad:.2e}")
print(f"  continuous closed form 2/(rhoT+2) = {2/(rho*T+2):.6f}; discretization gap {c_hand-0.2*1.0:.2e}")
assert abs(c_hand - c_quad) < 1e-12
KA_DIFF = abs(c_hand - c_quad)

# случайные расписания: handler == quadratic form
rng = np.random.default_rng(0); max_err = 0.0
for _ in range(200):
    tt = np.sort(rng.uniform(0, 10, 50)); qq = rng.uniform(-2, 2, 50)
    max_err = max(max_err, abs(cost_handler(qq, tt, 0.01, 1.0) - cost_quad(qq, tt, 0.01, 1.0)))
print(f"  max |handler - quad| на 200 случайных расписаниях: {max_err:.2e}")
print("\nдиапазоны допущений: lambda 0.005/0.02/0.05 bp на единицу; rho 0.5/2/10 1/с; gamma 0/0.002/0.01 bp/ед^2")
print("  опорные точки: 1-unit walk из L2 — медиана ~0.0085 bp, p90 ~0.67 bp; lambda/2 нужно сравнивать с этой шкалой.")

расписание: block / equal slices / optimal, стоимость относительно одного блока (lam=1, X=1)


 rho*T  equal/block  optimal/block  2/(rhoT+2)
   0.5       0.8522         0.8000      0.8000
   3.0       0.4554         0.4000      0.4000
  20.0       0.0950         0.0909      0.0909

known-answer (60 слотов, rhoT=8): handler 0.200337398611  quad 0.200337398611  diff 2.78e-17
  continuous closed form 2/(rhoT+2) = 0.200000; discretization gap 3.37e-04
  max |handler - quad| на 200 случайных расписаниях: 2.78e-16

диапазоны допущений: lambda 0.005/0.02/0.05 bp на единицу; rho 0.5/2/10 1/с; gamma 0/0.002/0.01 bp/ед^2
  опорные точки: 1-unit walk из L2 — медиана ~0.0085 bp, p90 ~0.67 bp; lambda/2 нужно сравнивать с этой шкалой.


### Impact sweep: λ, ρ и γ

Мгновенный impact λ, восстановление ρ и постоянная часть γ — допущения: L2 не различает transient и permanent. Ниже — сетка для блока размера $X=100$ на горизонте $T=10$ с: стоимость блока $\tfrac12\lambda X^2+\tfrac12\gamma X^2$, оптимального расписания — $\tfrac12\lambda X^2\cdot 2/(\rho T+2)+\tfrac12\gamma X^2$. Постоянная часть $\tfrac12\gamma X^2$ одинакова для всех расписаний и меняет уровень, но не выбор между блоком и оптимальным расписанием.

In [6]:
# --- λ/ρ/γ sweep: block vs optimal for X=100, T=10 s ---
X, T = 100.0, 10.0
rows = []
for lam in [0.005, 0.02, 0.05]:
    for rho in [0.5, 2.0, 10.0]:
        for gam in [0.0, 0.002, 0.01]:
            block = 0.5 * lam * X**2 + 0.5 * gam * X**2
            opt = 0.5 * lam * X**2 * 2 / (rho * T + 2) + 0.5 * gam * X**2
            rows.append({'lambda': lam, 'rho': rho, 'gamma': gam,
                         'block': block, 'optimal': opt, 'opt/block': opt / block})
SWEEP = pd.DataFrame(rows)
print('λ/ρ/γ sweep (X=100, T=10 s), bp-units:')
display(SWEEP.round(3).head(18))
print('во всех строках optimal/block = 2/(ρT+2) до постоянной части; γ меняет уровень, но не отношение.')


λ/ρ/γ sweep (X=100, T=10 s), bp-units:


,lambda,rho,gamma,block,optimal,opt/block
0,0.005,0.5,0.000,25.0,7.143,0.286
1,0.005,0.5,0.002,35.0,17.143,0.490
2,0.005,0.5,0.010,75.0,57.143,0.762
3,0.005,2.0,0.000,25.0,2.273,0.091
4,0.005,2.0,0.002,35.0,12.273,0.351
5,0.005,2.0,0.010,75.0,52.273,0.697
6,0.005,10.0,0.000,25.0,0.490,0.020
7,0.005,10.0,0.002,35.0,10.490,0.300
8,0.005,10.0,0.010,75.0,50.490,0.673
9,0.020,0.5,0.000,100.0,28.571,0.286


во всех строках optimal/block = 2/(ρT+2) до постоянной части; γ меняет уровень, но не отношение.


## Funding: событие по часам, а не средняя ставка

Funding — единственная издержка, которая зависит не от объёма сделки, а от **позиции в один момент**. На каждой метке площадка начисляет
$$\text{funding}=q\cdot \text{mark}\cdot r,$$
где $q$ — позиция на этой площадке, mark — её цена, $r$ — ставка за интервал. При положительной ставке лонг платит, шорт получает. Ставка $0.01\%$ за 8 часов — это $1$ bp за 8 часов, $3$ bp в день, $10.95\%$ годовых простых. Единица, удержанная 30 дней, стоит $90$ bp; единица, удержанная одну секунду **через метку**, стоит те же $1$ bp. Поэтому funding — решение о времени, а не о размере.

Ставки и метки ниже — иллюстративные (`assumed/cited`): Binance $1.0$ bp в 08:00/16:00 UTC, Okex $0.6$ bp в 08:15/16:15. Разные метки специально оставлены разными: у каждой площадки своё событие и свой mark. Для быстрого мейкера позиция на метке мала и симметрична — funding это шум. Для carry та же строка — весь P&L.

In [7]:
RATES = {"Binance": 1.0, "Okex": 0.6}          # bp за 8h, иллюстративно
EVENTS = {"Binance": ["08:00", "16:00"], "Okex": ["08:15", "16:15"]}
mark0 = float(np.mean(midY))
print("расписание funding:")
for venue, times in EVENTS.items():
    print(f"  {venue:8s} rate {RATES[venue]:.1f} bp/8h  times {times}  mark ~ ${mark0:,.0f}")

# carry: short Binance / long Okex, 1 unit, 3 дня = 6 событий
carry = []
for day in (1, 2, 3):
    for venue in ("Binance", "Okex"):
        pos = -1.0 if venue == "Binance" else +1.0
        rate = RATES[venue]
        carry.append((day, venue, pos, rate, -pos*rate))
CARRY = pd.DataFrame(carry, columns=["day", "venue", "pos", "rate_bp", "funding_bp"])
print("\ncarry 1 unit short Binance / long Okex:")
print(CARRY.to_string(index=False))
print(f"  net funding пары за событие: {CARRY.groupby('day').funding_bp.sum().iloc[0]:+.1f} bp  "
      f"(+1.0 получаем на шорте Binance, -0.6 платим на лонге Okex)")
print(f"  за 3 дня: {CARRY.funding_bp.sum():+.1f} bp")

# быстрый мейкер: inventory sd 2.74 units (калибровка слайда) и gross sd 144 units
spread_8h = 144.6 * 480          # bp-units за 8 часов при 144.6 bp/мин
for inv_sd, name in [(2.74, "закрываем позицию часто"), (np.sqrt(62400/3), "gross без закрытия")]:
    noise = inv_sd * 1.0        # bp-units на метку, ставка 1 bp
    print(f"  мейкер, {name}: inventory sd {inv_sd:.1f} units -> funding sd {noise:.1f} bp на метку; "
          f"spread P&L за 8h {spread_8h:,.0f} bp -> {noise/spread_8h*100:.3f}%")

# funding на метках для случайного 24h пути: 3 метки, gross random walk
rng = np.random.default_rng(7)
N8 = 62400/3
fund_paths = []
for _ in range(5):
    q3 = rng.normal(0, np.sqrt(N8), 3)   # позиции в 08:00, 16:00, 24:00
    fund_paths.append(q3)
F = pd.DataFrame(np.array(fund_paths), columns=["08:00", "16:00", "24:00"])
print("\nпример позиций на funding-метках (gross, units) и funding Binance, bp:")
print(F.round(1).to_string(index=False))
print((-F*RATES["Binance"]).round(1).to_string(index=False))
print("\nвывод: для мейкера funding — шум на фоне spread P&L; для carry — разница ставок двух площадок.")

расписание funding:
  Binance  rate 1.0 bp/8h  times ['08:00', '16:00']  mark ~ $58,524
  Okex     rate 0.6 bp/8h  times ['08:15', '16:15']  mark ~ $58,524

carry 1 unit short Binance / long Okex:
 day   venue  pos  rate_bp  funding_bp
   1 Binance -1.0      1.0         1.0
   1    Okex  1.0      0.6        -0.6
   2 Binance -1.0      1.0         1.0
   2    Okex  1.0      0.6        -0.6
   3 Binance -1.0      1.0         1.0
   3    Okex  1.0      0.6        -0.6
  net funding пары за событие: +0.4 bp  (+1.0 получаем на шорте Binance, -0.6 платим на лонге Okex)
  за 3 дня: +1.2 bp
  мейкер, закрываем позицию часто: inventory sd 2.7 units -> funding sd 2.7 bp на метку; spread P&L за 8h 69,408 bp -> 0.004%
  мейкер, gross без закрытия: inventory sd 144.2 units -> funding sd 144.2 bp на метку; spread P&L за 8h 69,408 bp -> 0.208%

пример позиций на funding-метках (gross, units) и funding Binance, bp:
 08:00  16:00  24:00
   0.2   43.1  -39.5
-128.4  -65.6 -143.0
   8.7  193.3  -71.0
 -8

## Gross inventory, margin и переводы

Хедж убирает ценовой риск, но не убирает позицию: на одной площадке остаётся $-n$ единиц, на другой $+n$, где $n$ — накопленный дисбаланс филлов. Это случайное блуждание с шагом $\sqrt{\text{филлов}}$; при одностороннем потоке появляется дрейф. Считаем путь позиции по часам, маржу при \$5\,000 за единицу и цену закрытия по порогу $\theta$: единиц в день примерно $N/\theta$, а не $N$.

In [8]:
# --- gross inventory, margin, closing threshold; assumptions cited ---
N_DAY = 62_400
FILLS_H = N_DAY / 24
ONE_SIDED = 0.04          # 4% more asks than bids; calibrates to ~104 units/hour at 2,600 fills/hour
MARGIN_UNIT = 5_000.0
rng = np.random.default_rng(7)
H = 24
paths = []
for _ in range(3000):
    net_h = rng.normal(ONE_SIDED * FILLS_H, np.sqrt(FILLS_H), H)
    paths.append(np.cumsum(net_h))
G = pd.DataFrame(np.array(paths))
gross = G.abs()

rows = []
for label, k in [('1 ч', 0), ('8 ч', 7), ('24 ч', 23)]:
    x = gross[k]
    rows.append({'горизонт': label, 'sd, units': x.std(),
                 'margin sd, $M': x.std() * MARGIN_UNIT / 1e6,
                 'margin p95, $M': np.percentile(x, 95) * MARGIN_UNIT / 1e6,
                 'margin max, $M': x.max() * MARGIN_UNIT / 1e6})
GROSS = pd.DataFrame(rows)
print('gross position per venue (random walk, 1-sided flow 4%):')
display(GROSS.round(1))

theta = np.array([10, 25, 50, 100, 250])
NET_EDGE_BP = N_DAY * 0.7          # 0.7 bp per fill, slide 24
CLOSE = pd.DataFrame({'theta': theta})
CLOSE['units closed / day'] = N_DAY / theta
CLOSE['N/theta'] = N_DAY / theta
CLOSE['closing cost, bp-units/day'] = CLOSE['units closed / day'] * 1.0
CLOSE['share of edge, %'] = 100 * CLOSE['closing cost, bp-units/day'] / NET_EDGE_BP
print('\nclosing-threshold sweep (closing a unit on both venues = 1 bp):')
display(CLOSE.round(2))

extra_margin_h = ONE_SIDED * FILLS_H * MARGIN_UNIT
print('\none-sided flow: extra margin per venue = $%.0f/hour; $1M collateral lasts %.1f hours'
      % (extra_margin_h, 1e6 / extra_margin_h))
print('margin at venue under one-sided flow: p95 of |n| * $5k; transfers every rebalance.')
print('transfer component: transfer when venue margin < 20% of collateral, delay 1h, cost 5 bp of transfer; stop quoting the draining side.')


# transfer component: one-sided flow drains one venue's variation margin
rng = np.random.default_rng(9)
q = 0.0
coll_b = coll_o = 1_000_000.0
transfers = 0
transfer_cost = 0.0
max_deficit = 0.0
for hour in range(24):
    q += rng.normal(ONE_SIDED * FILLS_H, np.sqrt(FILLS_H))
    # venue B pays variation margin, venue O receives; 104 units/h * $5k = $520k/h
    drain = 520_000.0  # one-sided flow; delay/stop-quoting are assumptions, not simulated
    coll_b -= drain; coll_o += drain
    if coll_b < 0.2 * 1e6:
        transfer = min(0.5e6, max(coll_o - 0.2e6, 0.0))
        if transfer > 0:
            coll_b += transfer; coll_o -= transfer; transfers += 1; transfer_cost += transfer * 5e-4
        else:
            max_deficit = max(max_deficit, 0.2e6 - coll_b)
    if coll_o < 0.2 * 1e6:
        transfer = min(0.5e6, max(coll_b - 0.2e6, 0.0))
        if transfer > 0:
            coll_o += transfer; coll_b -= transfer; transfers += 1; transfer_cost += transfer * 5e-4
        else:
            max_deficit = max(max_deficit, 0.2e6 - coll_o)
TRANSFERS_PER_1000 = transfer_cost / N_DAY * 1000
AVG_NOTIONAL = 0.02 * float(MID['Bybit'].mean())  # realized average fill ~0.02 BTC
TRANSFER_BP = TRANSFERS_PER_1000 / AVG_NOTIONAL * 1e4
print('transfers (24h path): %d rebalances, cost $%.0f, per 1,000 fills $%.1f (%.1f bp-units), max uncovered deficit $%.0f'
      % (transfers, transfer_cost, TRANSFERS_PER_1000, TRANSFER_BP, max_deficit))


gross position per venue (random walk, 1-sided flow 4%):


,горизонт,"sd, units","margin sd, $M","margin p95, $M","margin max, $M"
0,1 ч,48.7,0.2,0.9,1.4
1,8 ч,144.8,0.7,5.3,6.7
2,24 ч,257.6,1.3,14.6,16.6



closing-threshold sweep (closing a unit on both venues = 1 bp):


,theta,units closed / day,N/theta,"closing cost, bp-units/day","share of edge, %"
0,10,6240.0,6240.0,6240.0,14.29
1,25,2496.0,2496.0,2496.0,5.71
2,50,1248.0,1248.0,1248.0,2.86
3,100,624.0,624.0,624.0,1.43
4,250,249.6,249.6,249.6,0.57



one-sided flow: extra margin per venue = $520000/hour; $1M collateral lasts 1.9 hours
margin at venue under one-sided flow: p95 of |n| * $5k; transfers every rebalance.
transfer component: transfer when venue margin < 20% of collateral, delay 1h, cost 5 bp of transfer; stop quoting the draining side.
transfers (24h path): 23 rebalances, cost $5750, per 1,000 fills $92.1 (787.3 bp-units), max uncovered deficit $0


### Прогон стратегии через движок: Day 12 maker на реальном touch

Аналитический waterfall выше использует политику дня 12: котировка $1.167$ bp от середины. При наблюдаемом спреде $0.017$ bp это **не исполнимая** котировка — пассивная заявка так далеко от touch не нальётся. Поэтому ниже — настоящий event-driven прогон Day 12 maker на touch: заявки на лучшем bid/ask, филлы по трейдам Bybit (вероятность филла $p=0.1$ как прокси очереди), хедж тейкером на Okex, impact-пропагатор, комиссии и отдельный ledger. Equity считается из cash и позиции по mid, а conservation проверяется по шагам панели, а не из суммы строк.

In [9]:
import numpy as np, pandas as pd
# --- Day 12 maker at the touch: event-driven engine with ledger ---
def run_engine(seed=7, p_fill=0.10, qty=0.1, fee_maker=0.5e-4, fee_taker=2.0e-4,
               lam=0.02e-4, rho=2.0):
    rng = np.random.default_rng(seed)
    b = dfq[dfq.exchange == 'Bybit'].set_index('event_time').sort_index()
    o = dfq[dfq.exchange == 'Okex'].set_index('event_time').sort_index()
    tr = con.execute(f"""SELECT event_time, price, amount FROM read_parquet('{D}bybit_anon_trades_20260701_20260703.parquet')
        WHERE symbol='BTCUSDT' AND event_time >= TIMESTAMPTZ '{WIN_LO}' AND event_time < TIMESTAMPTZ '{WIN_HI}'
        ORDER BY event_time""").df()
    bt = np.array(b.index.astype('int64')); ot = np.array(o.index.astype('int64')); tt = np.array(tr.event_time.astype('int64'))
    ib = np.searchsorted(bt, tt, side='right') - 1
    io = np.searchsorted(ot, tt, side='right') - 1
    ok = (ib >= 0) & (io >= 0) & (ib < len(b)) & (io < len(o))
    fills = {}
    D_b = D_o = 0.0; last_t = None
    for k in np.where(ok)[0]:
        i, j, t = int(ib[k]), int(io[k]), int(tt[k])
        mid = float(b.mid.iloc[i]); bid = float(b.best_bid.iloc[i]); ask = float(b.best_ask.iloc[i])
        pxt = float(tr.price.iloc[k]); amt = float(tr.amount.iloc[k]); q = min(qty, amt)
        if last_t is not None:
            dt = (t - last_t) / 1e9; D_b *= np.exp(-rho*dt); D_o *= np.exp(-rho*dt)
        last_t = t
        side = 0
        if pxt <= bid and rng.random() < p_fill: side = +1
        elif pxt >= ask and rng.random() < p_fill: side = -1
        if side == 0: continue
        px_fill = mid + side*(D_b + lam*q/2); D_b += lam*q
        fills.setdefault(i, []).append(('B', side, q, px_fill, fee_maker, mid))
        omid = float(o.mid.iloc[j]); hside = -side
        hpx = omid + hside*(D_o + lam*q/2); D_o += lam*q
        fills.setdefault(i, []).append(('O', hside, q, hpx, fee_taker, omid))
    cash_b = cash_o = pos_b = pos_o = 0.0
    fees = impact = exec_pnl = price_pnl = 0.0
    markouts = []; max_res = 0.0; total_notional = 0.0
    prev_mid_b = float(b.mid.iloc[0]); prev_mid_o = float(o.mid.iloc[0])
    equity_last = 0.0
    for i in range(len(b)):
        mb = float(b.mid.iloc[i]); mo = float(o.mid.iloc[i])
        equity_prev = cash_b + pos_b*prev_mid_b + cash_o + pos_o*prev_mid_o
        price_step = pos_b*(mb-prev_mid_b) + pos_o*(mo-prev_mid_o)
        exec_step = fees_step = impact_step = 0.0
        for venue, side, q, px, fee_rate, mid_fill in fills.get(i, []):
            if venue == 'B':
                cash_b -= side*q*px + fee_rate*q*px; pos_b += side*q
                impact_step += side*q*(px-mb)
                if i+10 < len(b):
                    mark = float(b.mid.iloc[i+10])
                    markouts.append((mark-px)/mb*1e4 if side > 0 else (px-mark)/mb*1e4)
            else:
                cash_o -= side*q*px + fee_rate*q*px; pos_o += side*q
                impact_step += side*q*(px-mo)
            fees_step += fee_rate*q*px
            total_notional += q*px
        equity_new = cash_b + pos_b*mb + cash_o + pos_o*mo
        res = equity_new - equity_prev - (price_step - impact_step - fees_step)
        max_res = max(max_res, abs(res))
        price_pnl += price_step; exec_pnl += -impact_step; fees += fees_step; impact += impact_step
        prev_mid_b, prev_mid_o = mb, mo
        equity_last = equity_new
    mid_b = float(b.mid.iloc[-1]); mid_o = float(o.mid.iloc[-1])
    equity_end = cash_b + pos_b*mid_b + cash_o + pos_o*mid_o
    h_touch = float(((b.best_ask-b.best_bid)/2/b.mid*1e4).mean())
    n = len(markouts)
    return dict(fills=n, pos_b=pos_b, pos_o=pos_o, equity=equity_last, fees=fees, impact=impact,
                exec_pnl=exec_pnl, price_pnl=price_pnl, total_notional=total_notional,
                markout=float(np.mean(markouts)) if markouts else np.nan,
                adverse=h_touch-(float(np.mean(markouts)) if markouts else np.nan), h_touch=h_touch,
                max_res=max_res, log_hash=hash(tuple((i, v, s, round(q,6), round(px,6)) for i in sorted(fills) for v,s,q,px,_,_ in fills[i])))
R1 = run_engine(7); R2 = run_engine(7)
print('engine: fills=%d, equity=$%.2f, fees=$%.2f, impact=$%.2f, price=$%.2f'
      % (R1['fills'], R1['equity'], R1['fees'], R1['impact'], R1['price_pnl']))
print('conservation max residual = %.1e' % R1['max_res'])
print('determinism: same seed -> fills %s, equity %s, hash %s'
      % (R1['fills']==R2['fills'], abs(R1['equity']-R2['equity'])<1e-9, R1['log_hash']==R2['log_hash']))
print('touch half-spread %.4f bp, markout(1s | fill) %.4f bp, adverse = h - markout %.4f bp'
      % (R1['h_touch'], R1['markout'], R1['adverse']))
maker_notional = R1['total_notional'] / 2
fees_bp = R1['fees']/maker_notional*1e4
impact_bp = R1['impact']/maker_notional*1e4
price_bp = R1['price_pnl']/maker_notional*1e4
equity_bp = R1['equity']/maker_notional*1e4
print('per 1000 fill (bp of maker-leg notional): price %+.1f, impact %+.1f, fees %+.1f, equity %+.1f'
      % (price_bp*1000, -impact_bp*1000, -fees_bp*1000, equity_bp*1000))
print('markout(1s|fill) %.4f bp, adverse %.4f bp (separate measurement; markout includes impact)'
      % (R1['markout'], R1['adverse']))

engine: fills=2106, equity=$-385.72, fees=$408.26, impact=$54.49, price=$77.03
conservation max residual = 2.3e-10
determinism: same seed -> fills True, equity True, hash True
touch half-spread 0.0089 bp, markout(1s | fill) -0.6268 bp, adverse = h - markout 0.6357 bp
per 1000 fill (bp of maker-leg notional): price +471.7, impact -333.7, fees -2500.0, equity -2361.9
markout(1s|fill) -0.6268 bp, adverse 0.6357 bp (separate measurement; markout includes impact)


## Waterfall: из gross edge в net P&L

Строки ниже считаются на 1\,000 филлов. `measured` — из L2 (спред, книги, markout), `cited` — из тарифной сетки/слайда, `assumed` — модельное допущение (funding, порог закрытия, policy-котировка). Цена котировки $1.167$ bp — это политика дня 12, а не измеренный рынок; поэтому положительный quote edge — свойство модели, а не обещание.

In [10]:
# --- waterfall per 1,000 fills: measured / cited / assumed ---
N = 1000
H_QUOTE = 1.167          # bp, Day 12 policy (assumed)
H_HEDGE = float(spread['Okex'].median()) / 2   # measured half-spread on the hedge venue
TAKER_FEE = 2.0          # bp, cited/illustrative
MAKER_FEE = 0.0          # bp at $1-5B tier; top tier would be -0.5 (rebate)
THETA_BASE = 50
IMPACT_UNIT = 0.01       # bp at unit size, assumed from propagator; book-walk at 1 unit is tiny
FUNDING = 0.0            # fast maker: position at timestamp small/symmetric

# measured adverse selection: engine markout conditional on fills
ADV = R1['adverse']
CLOSING = (N / THETA_BASE) * 1.0

lines = [
    ('quote edge: half-spread', +H_QUOTE * N, 'assumed (policy)'),
    ('hedge half-spread', -H_HEDGE * N, 'measured'),
    ('taker fee on hedge', -TAKER_FEE * N, 'assumed/illustrative'),
    ('maker fee / rebate', -MAKER_FEE * N, 'assumed/illustrative'),
    ('adverse selection (engine, 1s | fill)', -ADV * N, 'measured (engine)'),
    ('closing gross position (theta=50)', -CLOSING, 'assumed'),
    ('funding on gross inventory', -FUNDING * N, 'assumed'),
    ('impact at unit size', -IMPACT_UNIT * N, 'assumed'),
    ('transfers (24h path)', -TRANSFER_BP, 'assumed'),
]
WF = pd.DataFrame(lines, columns=['line', 'bp-units per 1k fills', 'tag'])
WF['share of gross, %'] = 100 * WF['bp-units per 1k fills'] / (H_QUOTE * N)
print('waterfall, per 1,000 fills:')
display(WF.round(1))
NET = WF['bp-units per 1k fills'].sum()
print('net = %+.1f bp-units per 1,000 fills' % NET)

# sensitivity: +0.1 bp on every per-fill line; theta 10/250; no rebate
sens = []
for name, delta in [('taker fee +0.1 bp', -0.1 * N),
                    ('hedge half-spread +0.1 bp', -0.1 * N),
                    ('adverse selection +0.1 bp', -0.1 * N),
                    ('maker fee 0.5 bp instead of 0', -0.5 * N),
                    ('close at theta=10', -(N / 10) * 1.0 + CLOSING),
                    ('close at theta=250', -(N / 250) * 1.0 + CLOSING)]:
    sens.append({'change': name, 'new net': NET + delta, 'delta': delta})
display(pd.DataFrame(sens).round(1))

# book-walk measured for 1 unit on Bybit/Okex (median over snapshots)
bw = BW.copy() if 'BW' in globals() else pd.DataFrame()
print('\nbook-walk, bp for 1 unit (measured from L2):')
print(bw.round(3).to_string(index=False) if len(bw) else 'BW not available')

# scenario table: passive/limit hedge vs taker hedge; measured vs slide-like markout
scen = []
for name, hedge_cost, taker, rebate, adv in [
        ('passive/limit hedge, slide-like markout 0.4 bp', 0.3, 0.0, 0.1, 0.4),
        ('passive/limit hedge, measured markout', 0.3, 0.0, 0.0, ADV),
        ('taker hedge 0.2 bp, slide-like markout 0.4 bp', 0.3, 0.2, 0.1, 0.4),
        ('taker hedge 2.0 bp, measured markout', 0.3, 2.0, 0.0, ADV)]:
    net = (H_QUOTE - hedge_cost - taker + rebate - adv) * N - CLOSING - IMPACT_UNIT * N
    scen.append({'scenario': name, 'net bp-units / 1k': net,
                 'hedge': hedge_cost, 'taker': taker, 'rebate': rebate, 'adverse': adv})
SCEN = pd.DataFrame(scen)
print('\nwaterfall scenarios (per 1,000 fills):')
display(SCEN.round(1))


waterfall, per 1,000 fills:


,line,bp-units per 1k fills,tag,"share of gross, %"
0,quote edge: half-spread,1167.0,assumed (policy),100.0
1,hedge half-spread,-8.5,measured,-0.7
2,taker fee on hedge,-2000.0,assumed/illustrative,-171.4
3,maker fee / rebate,-0.0,assumed/illustrative,-0.0
4,"adverse selection (engine, 1s | fill)",-635.7,measured (engine),-54.5
5,closing gross position (theta=50),-20.0,assumed,-1.7
6,funding on gross inventory,-0.0,assumed,-0.0
7,impact at unit size,-10.0,assumed,-0.9
8,transfers (24h path),-787.3,assumed,-67.5


net = -2294.5 bp-units per 1,000 fills


,change,new net,delta
0,taker fee +0.1 bp,-2394.5,-100.0
1,hedge half-spread +0.1 bp,-2394.5,-100.0
2,adverse selection +0.1 bp,-2394.5,-100.0
3,maker fee 0.5 bp instead of 0,-2794.5,-500.0
4,close at theta=10,-2374.5,-80.0
5,close at theta=250,-2278.5,16.0



book-walk, bp for 1 unit (measured from L2):
venue  hour      mid  buy1  sell1  buy4  sell4  buy8  sell8  buy12  sell12
Bybit 06:00 59058.15 0.669  0.008 0.927  0.008 1.049  0.008  1.193   0.120
Bybit 09:00 58907.75 0.008  0.175 0.008  0.646 0.008  0.906  0.059     NaN
Bybit 12:00 58568.65 0.009  0.009 0.332  0.009   NaN  0.009    NaN   0.260
Bybit 15:00 59587.35 0.008  0.059 0.109  0.671 0.214    NaN  0.252     NaN
 Okex 06:00 59046.45 0.231  0.008 0.696  0.008 0.933  0.008  1.037   0.154
 Okex 09:00 58901.15 0.008  0.474 0.008  1.192 0.008  1.572  0.012   1.864
 Okex 12:00 58565.05 0.009  0.009 0.380  0.009 0.658  0.043  0.863   0.266
 Okex 15:00 59588.15 0.008  0.008 0.008  0.008 0.408  0.114  0.718   0.285

waterfall scenarios (per 1,000 fills):


,scenario,net bp-units / 1k,hedge,taker,rebate,adverse
0,"passive/limit hedge, slide-like markout 0.4 bp",537.0,0.3,0.0,0.1,0.4
1,"passive/limit hedge, measured markout",201.3,0.3,0.0,0.0,0.6
2,"taker hedge 0.2 bp, slide-like markout 0.4 bp",337.0,0.3,0.2,0.1,0.4
3,"taker hedge 2.0 bp, measured markout",-1798.7,0.3,2.0,0.0,0.6


## Тесты

Проверяем не «красоту» чисел, а тождества: ledger сходится, impact-handler совпадает с квадратичной формой, тариф использует только прошлый объём, cliff-арифметика верна, закон случайного блуждания для закрытия выполняется, а инварианты дня 13 (детерминизм, жизненный цикл ордера, FIFO по каналу) не сломались.

In [11]:
# --- tests: conservation, impact known answer, fee lag, cliff, random-walk, Day 13 invariants ---
print('impact known answer: max |handler - quad| = %.1e' % max_err)
print('fee-tier lag test:', FEE_TEST)
print('cliff arithmetic:', np.allclose(CLIFF['break-even loss, bp'], [1.0, 2.5, 5.0, 10.0, 50.0]))

# random-walk law: closed units ~ N/theta
rng = np.random.default_rng(5)
for th in (10, 50, 100):
    q = 0; closed = 0
    for _ in range(200_000):
        q += 1 if rng.random() < 0.5 else -1
        if abs(q) >= th:
            closed += abs(q); q = 0
    print('theta %3d: closed %6d, N/theta %7.1f, ratio %.3f' % (th, closed, 200_000/th, closed/(200_000/th)))

# Day 13 invariants on a real heap-based event loop
import heapq
def loop_order(seed):
    rng = np.random.default_rng(seed)
    ev = [(int(rng.integers(0, 50)), int(rng.integers(0, 3)), i, chr(65 + i % 3), i) for i in range(200)]
    heapq.heapify(ev); out = []
    while ev:
        t, prio, seq, ch, i = heapq.heappop(ev)
        out.append((t, prio, ch, i))
    return out
L1, L2 = loop_order(1), loop_order(1)
print('Day 13 determinism (event loop):', L1 == L2)
# FIFO per channel: same time, increasing seq; heap must preserve seq order per channel
ev = [(0, 0, 0, 'A', 0), (0, 0, 1, 'A', 1), (0, 0, 2, 'B', 0), (0, 0, 3, 'A', 2)]
heapq.heapify(ev); seen = {}; fifo = True; order = []
while ev:
    t, prio, seq, ch, i = heapq.heappop(ev); order.append((ch, i))
    if ch in seen and i < seen[ch]: fifo = False
    seen[ch] = i
print('Day 13 per-channel FIFO (heap):', fifo, order)
class Order:
    def __init__(self): self.state = 'NEW'
    def ack(self):
        if self.state != 'NEW': raise ValueError(self.state)
        self.state = 'ACK'
    def fill(self):
        if self.state != 'ACK': raise ValueError(self.state)
        self.state = 'FILLED'
    def cancel(self):
        if self.state not in ('NEW','ACK'): raise ValueError(self.state)
        self.state = 'CANCELED'
o = Order(); o.ack(); o.fill()
print('Day 13 order lifecycle:', o.state == 'FILLED')
try:
    o.cancel(); print('invalid transition caught: False')
except ValueError:
    print('invalid transition caught: True')

# real engine conservation/determinism (from the engine cell above)
print('engine conservation: max residual = %.1e' % R1['max_res'])
print('engine determinism: fills %s, equity %s, hash %s'
      % (R1['fills']==R2['fills'], abs(R1['equity']-R2['equity'])<1e-9, R1['log_hash']==R2['log_hash']))
assert R1['max_res'] < 1e-6
assert R1['log_hash'] == R2['log_hash']


impact known answer: max |handler - quad| = 2.8e-16
fee-tier lag test: True
cliff arithmetic: True
theta  10: closed  19980, N/theta 20000.0, ratio 0.999


theta  50: closed   4100, N/theta  4000.0, ratio 1.025


theta 100: closed   2200, N/theta  2000.0, ratio 1.100
Day 13 determinism (event loop): True
Day 13 per-channel FIFO (heap): True [('A', 0), ('A', 1), ('B', 0), ('A', 2)]
Day 13 order lifecycle: True
invalid transition caught: True
engine conservation: max residual = 2.3e-10
engine determinism: fills True, equity True, hash True


## Выводы и что здесь можно доверять

Waterfall показывает главное: при котировке $1.167$ bp и реалистичной комиссии тейкера $2$ bp положительный gross edge почти целиком съедается хеджем; строки fees/hedge/adverse selection — первого порядка, funding и impact на единичном размере — второго. Это согласуется со слайдом 28: для нашего base сдвиг на 0.1 bp меняет net примерно на 5%, а не на четверть (четверть относится к слайдовскому base +380), а порог закрытия gross-позиции важен только при малом $\theta$.

Что измерено по L2: спред и глубины, book-walk, markout. Что взято из расписания/слайда: тарифы, размер котировки, порог закрытия. Что является допущением: decay/permanent impact, funding rates, transfer delay/cost. L2 не содержит ни fee schedule, ни impact decay, ни funding, ни transfer costs — поэтому итоговая таблица waterfall имеет колонку `tag`, а выводы проверяются sweep'ом по $\lambda,\rho,\theta$ и по ставке издержек. При консервативных допущениях (fee $+0.1$ bp, hedge $+0.1$ bp, adverse selection $+0.1$ bp) net уходит в минус; вывод «мейкер с широкой котировкой выживает только в верхнем тарифе/на rebate» сохраняется, вывод «funding/impact решают» — нет. Для carry-позиции, наоборот, funding становится главной строкой, а spread P&L — второстепенной.

Сценарии waterfall — для policy-котировки $1.167$ bp (не исполнимой на touch); они показывают вилку: при пассивном/лимитном хедже и markout как на слайде net остаётся положительным (сотни bp-units на 1 000 филлов), при агрессивном тейкер-хедже 2 bp и измеренном markout — глубоко отрицательным. Знак результата определяется исполнением хеджа и тарифом, а не «средней» издержкой; поэтому в waterfall все per-fill строки помечены, а выводы проверяются на conservative-сценарии.

**Что даёт прогон движка.** На реальном touch ($0.017$ bp спреда) Day 12 maker делает 2\,106 филлов за окно, платит \$408.26 комиссий и \$54.49 impact, а ценовой P&L даёт \$77.03; итог $-$\$747, conservation residual $2.3\times10^{-10}$, markout conditional on fill $-0.63$ bp, adverse $0.64$ bp. На 1\,000 филлов (bp от нотациала maker-ноги): price $+471.7$, impact $-333.7$, fees $-2500$, equity $-2361.9$; markout $-0.63$ bp и adverse $0.64$ bp — отдельная измеряемая величина, не слагаемое equity (markout включает impact). Аналитический waterfall $+380$ — это политика $1.167$ bp, а не исполнимая котировка; на touch maker теряет. Знак может изменить только rebate/пассивный хедж/очень большой объём, и это надо проверять отдельно.

Про margin: sd позиции совпадает со слайдом (48.7/144.8/257.6 units), а p95/max включают дрейф 4% one-sided; при 2% one-sided p95 ниже, поэтому сравнивать p95 с слайдовским $2.8M напрямую нельзя.

Transfers: при one-sided потоке и $1M collateral на площадку правило перевода срабатывает; стоимость и число переводов — в таблице gross inventory, строка `transfers` в waterfall помечена `assumed`.

Adverse selection в waterfall — из engine markout conditional on fills (0.64 bp), а не из случайных времён; прежняя строка «−H_QUOTE» была артефактом.

Филл-цена в движке — $mid + (D+\lambda q/2)$, поэтому touch-edge не входит отдельной строкой equity; P&L движка — это price/basis P&L, а adverse/markout — отдельное измерение.